In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/WELFake_Dataset.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df = df[~((df["title"] == "") & (df["text"] == ""))].copy()

df["content"] = df["title"] + " " + df["text"]

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df["content_clean"] = df["content"].apply(clean_text)

print("Shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())

Shape: (72134, 5)
Label distribution:
 label
1    37106
0    35028
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

X_all = df["content_clean"]
y_all = df["label"]

X_sample, _, y_sample, _ = train_test_split(
    X_all, y_all,
    train_size=10000,
    random_state=42,
    stratify=y_all
)

df_sample = pd.DataFrame({"text": X_sample.values, "label": y_sample.values})

df_fake = df_sample[df_sample["label"] == 0].copy()
df_real = df_sample[df_sample["label"] == 1].copy()

print("Sample size:", len(df_sample))
print("Fake:", len(df_fake), "Real:", len(df_real))

Sample size: 10000
Fake: 4856 Real: 5144


In [3]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

def to_transaction(text, max_tokens=50):
    tokens = text.split()
    tokens = [t for t in tokens if (t not in stopwords and len(t) >= 3)]
    # uzmi prvih max_tokens radi kontrole veličine transakcija
    return list(dict.fromkeys(tokens[:max_tokens]))  # uniq + zadrži redosled

df_sample["transaction"] = df_sample["text"].apply(to_transaction)
df_fake["transaction"] = df_fake["text"].apply(to_transaction)
df_real["transaction"] = df_real["text"].apply(to_transaction)

print(df_sample["transaction"].head())

0    [alicia, machado, sex, tape, trump, just, trie...
1    [revealed, dark, agenda, globalization, open, ...
2    [little, boy, humiliates, sick, liberal, mom, ...
3    [wow, happened, somebody, asked, beyonces, rac...
4    [trump, rolls, obamaera, education, regulation...
Name: transaction, dtype: object


In [4]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(df_sample["transaction"]).transform(df_sample["transaction"])
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

print("One-hot shape:", df_onehot.shape)
df_onehot.head()

One-hot shape: (10000, 38752)


,aaa,aab,aamaq,aand,aaron,aarp,aba,ababa,aback,abadi,...,zurich,zurichs,zusammengekratzt,zuvor,zvezda,zvulun,zwak,zweck,zwickau,zzzzzzzz
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [5]:
from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(
    df_onehot,
    min_support=0.01,
    use_colnames=True
)

print("Broj frequent itemsets:", len(frequent_itemsets))

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

rules = rules.sort_values(by="lift", ascending=False)

print("Broj pravila:", len(rules))
rules.head(10)

Broj frequent itemsets: 2657
Broj pravila: 2376


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
870,"(james, fbi)","(comey, director)",0.0118,0.0126,0.0106,0.898305,71.294054,1.0,0.010451,9.709433,0.997747,0.768116,0.897007,0.869787
867,"(comey, director)","(james, fbi)",0.0126,0.0118,0.0106,0.841270,71.294054,1.0,0.010451,6.225660,0.998555,0.768116,0.839374,0.869787
872,(comey),"(james, director, fbi)",0.0139,0.0112,0.0106,0.762590,68.088386,1.0,0.010444,4.164945,0.999202,0.731034,0.759901,0.854509
865,"(james, director, fbi)",(comey),0.0112,0.0139,0.0106,0.946429,68.088386,1.0,0.010444,18.407200,0.996474,0.731034,0.945673,0.854509
207,(comey),"(james, fbi)",0.0139,0.0118,0.0111,0.798561,67.674674,1.0,0.010936,4.905707,0.999111,0.760274,0.796156,0.869620
206,"(james, fbi)",(comey),0.0118,0.0139,0.0111,0.940678,67.674674,1.0,0.010936,16.622829,0.996988,0.760274,0.939842,0.869620
62,(putin),(vladimir),0.0137,0.0116,0.0105,0.766423,66.070979,1.0,0.010341,4.231588,0.998545,0.709459,0.763682,0.835798
61,(vladimir),(putin),0.0116,0.0137,0.0105,0.905172,66.070979,1.0,0.010341,10.400982,0.996423,0.709459,0.903855,0.835798
868,"(comey, fbi)","(james, director)",0.0125,0.0131,0.0106,0.848000,64.732824,1.0,0.010436,6.492763,0.997015,0.706667,0.845982,0.828580
869,"(james, director)","(comey, fbi)",0.0131,0.0125,0.0106,0.809160,64.732824,1.0,0.010436,5.174500,0.997621,0.706667,0.806745,0.828580


---

## Association rules – ceo uzorak

Primena Apriori algoritma sa minimalnim support-om od 1% rezultovala je:

- 2657 frequent itemset-a
- 2376 pravila pridruživanja

Najviša lift vrednost primećena je kod pravila koja uključuju lična imena i njihove funkcije, npr:

- (james, fbi) → (comey, director)
- (putin) → (vladimir)

---

### Interpretacija

Visoke lift vrednosti (> 60) ukazuju na veoma snažnu povezanost između termina koji se gotovo uvek javljaju zajedno.

Ovakva pravila reflektuju činjenicu da su tekstualni podaci bogati entitetima (imena, prezimena, titule), i da se ti entiteti pojavljuju u stabilnim kombinacijama.

Međutim, ova pravila više odražavaju ko-pojavljivanje entiteta nego klasnu razliku između fake i real vesti.

In [6]:
te_fake = TransactionEncoder()
te_fake_ary = te_fake.fit(df_fake["transaction"]).transform(df_fake["transaction"])
df_fake_onehot = pd.DataFrame(te_fake_ary, columns=te_fake.columns_)

print("Fake one-hot shape:", df_fake_onehot.shape)

Fake one-hot shape: (4856, 23183)


In [7]:
frequent_fake = apriori(
    df_fake_onehot,
    min_support=0.02,
    use_colnames=True
)

rules_fake = association_rules(
    frequent_fake,
    metric="confidence",
    min_threshold=0.6
)

rules_fake = rules_fake.sort_values(by="lift", ascending=False)

print("Fake rules:", len(rules_fake))
rules_fake.head(10)

Fake rules: 2525


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
824,"(donald, clinton)","(trump, hillary)",0.028624,0.030478,0.026359,0.920863,30.214272,1.0,0.025487,12.251236,0.995396,0.805031,0.918376,0.892864
821,"(trump, hillary)","(donald, clinton)",0.030478,0.028624,0.026359,0.864865,30.214272,1.0,0.025487,7.188180,0.997298,0.805031,0.860883,0.892864
823,"(donald, hillary)","(trump, clinton)",0.030272,0.029036,0.026359,0.870748,29.988324,1.0,0.025480,7.512193,0.996830,0.800000,0.866883,0.889275
822,"(trump, clinton)","(donald, hillary)",0.029036,0.030272,0.026359,0.907801,29.988324,1.0,0.025480,10.517821,0.995561,0.800000,0.904923,0.889275
815,"(clinton, presidential)","(hillary, democratic)",0.030066,0.025329,0.020387,0.678082,26.770464,1.0,0.019626,3.027700,0.992485,0.582353,0.669716,0.741480
812,"(hillary, democratic)","(clinton, presidential)",0.025329,0.030066,0.020387,0.804878,26.770464,1.0,0.019626,4.970912,0.987662,0.582353,0.798830,0.741480
814,"(clinton, democratic)","(hillary, presidential)",0.024918,0.031713,0.020387,0.818182,25.799292,1.0,0.019597,5.325577,0.985803,0.562500,0.812227,0.730519
813,"(hillary, presidential)","(clinton, democratic)",0.031713,0.024918,0.020387,0.642857,25.799292,1.0,0.019597,2.730231,0.992722,0.562500,0.633731,0.730519
829,"(reuters, hillary)","(clinton, presidential)",0.028007,0.030066,0.021005,0.750000,24.945205,1.0,0.020163,3.879736,0.987571,0.566667,0.742251,0.724315
832,"(clinton, presidential)","(reuters, hillary)",0.030066,0.028007,0.021005,0.698630,24.945205,1.0,0.020163,3.225251,0.989667,0.566667,0.689947,0.724315


In [8]:
te_real = TransactionEncoder()
te_real_ary = te_real.fit(df_real["transaction"]).transform(df_real["transaction"])
df_real_onehot = pd.DataFrame(te_real_ary, columns=te_real.columns_)

print("Real one-hot shape:", df_real_onehot.shape)

Real one-hot shape: (5144, 28534)


In [9]:
frequent_real = apriori(
    df_real_onehot,
    min_support=0.02,
    use_colnames=True
)

rules_real = association_rules(
    frequent_real,
    metric="confidence",
    min_threshold=0.6
)

rules_real = rules_real.sort_values(by="lift", ascending=False)

print("Real rules:", len(rules_real))
rules_real.head(10)

Real rules: 93


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,(century),(wire),0.029355,0.024883,0.020995,0.715232,28.743377,1.0,0.020265,3.424247,0.994400,0.631579,0.707965,0.779491
3,(wire),(century),0.024883,0.029355,0.020995,0.843750,28.743377,1.0,0.020265,6.212131,0.989840,0.631579,0.839025,0.779491
85,"(trump, states)",(united),0.026633,0.055988,0.020607,0.773723,13.819546,1.0,0.019115,4.171926,0.953020,0.332288,0.760303,0.570889
86,"(trump, united)",(states),0.025078,0.062792,0.020607,0.821705,13.086231,1.0,0.019032,5.256517,0.947341,0.306358,0.809760,0.574939
91,"(donald, hillary)","(trump, clinton)",0.044129,0.057932,0.033243,0.753304,13.003341,1.0,0.030686,3.818742,0.965713,0.483051,0.738134,0.663565
18,(states),(united),0.062792,0.055988,0.044323,0.705882,12.607843,1.0,0.040808,3.209642,0.982369,0.595300,0.688439,0.748775
19,(united),(states),0.055988,0.062792,0.044323,0.791667,12.607843,1.0,0.040808,4.498600,0.975288,0.595300,0.777709,0.748775
92,"(donald, clinton)","(trump, hillary)",0.039658,0.069790,0.033243,0.838235,12.010814,1.0,0.030475,5.750389,0.954599,0.436224,0.826099,0.657279
80,"(president, white)",(house),0.027216,0.083981,0.022551,0.828571,9.866138,1.0,0.020265,5.343442,0.923785,0.254386,0.812855,0.548545
81,"(president, house)",(white),0.027799,0.091563,0.022551,0.811189,8.859353,1.0,0.020005,4.811352,0.912492,0.232932,0.792158,0.528737


---

## Association rules – poređenje Fake i Real vesti

Analiza association pravila izvršena je posebno nad podskupovima Fake i Real vesti.

---

### Fake vesti

Broj pravila: 2525

Najizraženija pravila uključuju kombinacije političkih imena i termina:

- (donald, clinton) → (trump, hillary)
- (hillary, democratic) ↔ (clinton, presidential)
- (reuters, hillary) → (clinton, presidential)

Ova pravila ukazuju na snažno ko-pojavljivanje političkih aktera i izbornih termina.

---

### Real vesti

Broj pravila: 93

Najizraženija pravila uključuju institucionalne i formalne termine:

- (states) ↔ (united)
- (president, white) → (house)
- (wire) ↔ (century)

Ova pravila reflektuju standardizovane izraze i formalni novinarski stil.

---

### Ključne razlike

- Fake vesti generišu znatno veći broj pravila
- Fake pravila su dominantno politički obojena
- Real vesti pokazuju stabilne institucionalne fraze
- Lift vrednosti su visoke u oba slučaja, ali tematska struktura se razlikuje

## Zaključak:

Association analiza sugeriše da Fake vesti karakteriše intenzivno ponavljanje političkih entiteta i kombinacija imena, dok Real vesti pokazuju stabilniju i formalniju terminologiju.

In [10]:
from mlxtend.frequent_patterns import fpgrowth

frequent_fp = fpgrowth(
    df_onehot,
    min_support=0.01,
    use_colnames=True
)

rules_fp = association_rules(
    frequent_fp,
    metric="confidence",
    min_threshold=0.6
)

rules_fp = rules_fp.sort_values(by="lift", ascending=False)

print("FP-Growth rules:", len(rules_fp))
rules_fp.head(10)

FP-Growth rules: 2376


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2172,"(comey, director)","(james, fbi)",0.0126,0.0118,0.0106,0.841270,71.294054,1.0,0.010451,6.225660,0.998555,0.768116,0.839374,0.869787
2175,"(james, fbi)","(comey, director)",0.0118,0.0126,0.0106,0.898305,71.294054,1.0,0.010451,9.709433,0.997747,0.768116,0.897007,0.869787
2177,(comey),"(james, director, fbi)",0.0139,0.0112,0.0106,0.762590,68.088386,1.0,0.010444,4.164945,0.999202,0.731034,0.759901,0.854509
2170,"(james, director, fbi)",(comey),0.0112,0.0139,0.0106,0.946429,68.088386,1.0,0.010444,18.407200,0.996474,0.731034,0.945673,0.854509
2165,"(james, fbi)",(comey),0.0118,0.0139,0.0111,0.940678,67.674674,1.0,0.010936,16.622829,0.996988,0.760274,0.939842,0.869620
2166,(comey),"(james, fbi)",0.0139,0.0118,0.0111,0.798561,67.674674,1.0,0.010936,4.905707,0.999111,0.760274,0.796156,0.869620
2375,(putin),(vladimir),0.0137,0.0116,0.0105,0.766423,66.070979,1.0,0.010341,4.231588,0.998545,0.709459,0.763682,0.835798
2374,(vladimir),(putin),0.0116,0.0137,0.0105,0.905172,66.070979,1.0,0.010341,10.400982,0.996423,0.709459,0.903855,0.835798
2174,"(james, director)","(comey, fbi)",0.0131,0.0125,0.0106,0.809160,64.732824,1.0,0.010436,5.174500,0.997621,0.706667,0.806745,0.828580
2173,"(comey, fbi)","(james, director)",0.0125,0.0131,0.0106,0.848000,64.732824,1.0,0.010436,6.492763,0.997015,0.706667,0.845982,0.828580


---

## Poređenje Apriori i FP-Growth algoritama

Primena FP-Growth algoritma sa istim parametrima (min_support = 0.01, min_confidence = 0.6) rezultovala je:

- 2376 pravila pridruživanja

Broj pravila i dobijena pravila identični su rezultatima Apriori algoritma.

---

### Interpretacija

Apriori i FP-Growth algoritmi generišu isti skup frequent itemset-ova i pravila kada se primene nad istim podacima i parametrima.

Razlika između algoritama leži u implementaciji i efikasnosti:

- Apriori koristi generisanje kandidata i više prolaza kroz podatke
- FP-Growth koristi kompresovanu strukturu podataka (FP-tree)

U ovom eksperimentu oba algoritma daju identične rezultate, što potvrđuje konzistentnost analize.

## Završni zaključci analize pravila pridruživanja

Analiza pravila pridruživanja pokazala je:

- Snažno ko-pojavljivanje političkih imena i termina
- Znatno veći broj pravila u Fake vestima u odnosu na Real
- Tematske razlike između Fake i Real podskupova

Fake vesti karakteriše intenzivno ponavljanje političkih aktera i izbornih termina.

Real vesti pokazuju stabilne institucionalne fraze i formalni novinarski stil.

Rezultati ukazuju da struktura ko-pojavljivanja termina razlikuje Fake i Real vesti, ali ta razlika se više odnosi na tematski sadržaj nego na eksplicitnu klasnu separaciju.